In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()
GROQAI_API_KEY = os.getenv("GROQAI_API_KEY")

loader=PyPDFLoader("/Users/ysyseom/pdf-bot/attention_is_all_you_need.pdf")
pages = loader.load()

/var/folders/3t/w90p5wt14m17gr1007xzm2p40000gn/T/ipykernel_33323/3379834162.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/ysyseom/Library/Caches/pypoetry/virtualenvs/pdf-bot-R82_slq2-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
len(pages)

15

In [3]:
pages[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/ysyseom/pdf-bot/attention_is_all_you_need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\

In [4]:
pages[0].page_content

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurren

In [5]:
pages[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2023-08-03T00:07:29+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2023-08-03T00:07:29+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': '/Users/ysyseom/pdf-bot/attention_is_all_you_need.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap = 200,
)

In [7]:
splits = text_splitter.split_documents(pages)

In [8]:
len(splits)

52

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma.from_documents(splits, embeddings_model)

query= "What is the attention mechanism in transformer?"

docs=db.similarity_search(query)

print(docs[0].page_content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6640.03it/s]


The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward


In [10]:
print(docs[0].metadata)

{'author': '', 'producer': 'pdfTeX-1.40.25', 'source': '/Users/ysyseom/pdf-bot/attention_is_all_you_need.pdf', 'moddate': '2023-08-03T00:07:29+00:00', 'creationdate': '2023-08-03T00:07:29+00:00', 'page_label': '5', 'total_pages': 15, 'creator': 'LaTeX with hyperref', 'page': 4, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'keywords': '', 'trapped': '/False', 'subject': '', 'title': ''}


In [11]:
len(docs)

4

In [12]:
retriever = db.as_retriever()

In [13]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x148034590>, search_kwargs={})

In [14]:
#prompt
from langchain_core.prompts import ChatPromptTemplate

template="""Answer the question based only on the following context:
            <context>
            {context}
            </context>
            Question: {input}
            """
prompt = ChatPromptTemplate.from_template(template)

In [15]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n            <context>\n            {context}\n            </context>\n            Question: {input}\n            '), additional_kwargs={})])

In [19]:
from langchain_groq import ChatGroq
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQAI_API_KEY)

document_chain = create_stuff_documents_chain(model, prompt)
retriever_chain = create_retrieval_chain(retriever, document_chain)

response = retriever_chain.invoke({"input":"What is the attention mechanism in transformers?"})